# Part 1: Reasoning with ABA
ABA generator (class `ABAFramework`): literals, rules, assumptions, contraries, conversion to non-circular and atomic frameworks, arguments, attacks, and preferences (normal / reverse attacks).

Run the cells from top to bottom. Only Python 3 and its standard library are needed.

## 1. The generator

In [1]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Rule:
    # rule : head <- body (body can be empty)
    head: str
    body: tuple = ()

    def __str__(self):
        # empty body -> only the head
        if not self.body:
            return f"{self.head} <-"
        return f"{self.head} <- {', '.join(self.body)}"

class ABAFramework:
    def __init__(self):
        self.literals = set()      # language l
        self.rules = []            # list of Rule
        self.assumptions = set()   # a
        self.contraries = {}       # assumption -> its contrary
        self.preferences = set()   # pairs (weaker, stronger)

    # ---------- 1 & 2. definition of the framework ----------

    def add_literal(self, lit):
        self.literals.add(lit)

    def add_rule(self, head, body=()):
        self.rules.append(Rule(head, tuple(body)))

    def add_assumption(self, a, contrary):
        # an assumption and its contrary are both literals
        self.literals.add(a)
        self.literals.add(contrary)
        self.assumptions.add(a)
        self.contraries[a] = contrary

    def add_preference(self, weaker, stronger):
        # weaker < stronger
        self.preferences.add((weaker, stronger))

    # ---------- 3. conversion ----------

    def copy_without_rules(self):
        # new framework with same l, a, contraries and preferences, but no rules
        new = ABAFramework()
        new.literals = set(self.literals)
        new.assumptions = set(self.assumptions)
        new.contraries = dict(self.contraries)
        new.preferences = set(self.preferences)
        return new

    def to_non_circular(self):
        non_assump = self.literals - self.assumptions
        k = len(non_assump) # number of levels

        def name(s, i):
            # last level keeps the original name
            return s if i == k else f"{s}_{i}"

        new = self.copy_without_rules()
        # add the copies of each non assumption (one per level)
        for s in non_assump:
            for i in range(1, k + 1):
                new.literals.add(name(s, i))

        for rule in self.rules:
            atomic = set(rule.body) <= self.assumptions
            start = 1 if atomic else 2 # atomic rules: levels 1..k, others: 2..k
            for i in range(start, k + 1):
                if atomic:
                    new_body = rule.body
                else:
                    # assumptions stay, other literals come from the level below
                    new_body = tuple(b if b in self.assumptions else name(b, i - 1)
                                     for b in rule.body)
                new.rules.append(Rule(name(rule.head, i), new_body))
        return new

    def to_atomic(self):
        non_assump = self.literals - self.assumptions
        new = self.copy_without_rules()
        # two new assumptions for each non assumption
        for s in non_assump:
            new.literals |= {s + "_d", s + "_nd"}
            new.assumptions |= {s + "_d", s + "_nd"}
            new.contraries[s + "_d"] = s + "_nd"
            new.contraries[s + "_nd"] = s
        # in each body, a non assumption s becomes s_d
        for rule in self.rules:
            new_body = tuple(b if b in self.assumptions else b + "_d" for b in rule.body)
            new.rules.append(Rule(rule.head, new_body))
        return new

    def convert(self):
        # first remove cycles, then make atomic
        return self.to_non_circular().to_atomic()

    # ---------- 4. arguments and attacks ----------

    def build_arguments(self):
        # an argument is (support, conclusion)
        # every assumption is an argument for itself
        args = {(frozenset([a]), a) for a in self.assumptions}
        changed = True
        while changed: # repeat until nothing new
            changed = False
            for rule in self.rules:
                # for each literal of the body, all the supports that prove it
                options = [[X for (X, s) in args if s == b] for b in rule.body]
                # pick one support per body literal and merge them
                combos = [frozenset()]
                for opt in options:
                    combos = [c | X for c in combos for X in opt]
                for X in combos:
                    if (X, rule.head) not in args:
                        args.add((X, rule.head))
                        changed = True
        return args

    def build_attacks(self, args):
        # (x, s) attacks (y, t) if s is the contrary of an assumption in y
        attacks = set()
        for (X, s) in args:
            for (Y, t) in args:
                if any(self.contraries[a] == s for a in Y):
                    attacks.add(((X, s), (Y, t)))
        return attacks

    # ---------- 5. preferences: normal / reverse attacks ----------

    def build_attacks_plus(self, args):
        normal, reverse = set(), set()
        for (X, s) in args:
            for (Y, t) in args:
                for y in Y:
                    if self.contraries[y] == s: # (x, s) attacks y
                        # is one assumption of x weaker than the attacked y ?
                        if any((x, y) in self.preferences for x in X):
                            reverse.add(((Y, t), (X, s))) # attack is reversed
                        else:
                            normal.add(((X, s), (Y, t))) # normal attack
        return normal, reverse

        # ---------- display of the framework and its properties (for tests) ----------

    def is_atomic(self):
        # atomic if the body of every rule only contains assumptions
        for rule in self.rules:
            if not set(rule.body) <= self.assumptions:
                return False
        return True

    def is_circular(self):
        # circular if a literal can depend on itself through the rules
        # deps[h] = literals that appear in the bodies of the rules with head h
        deps = {}
        for rule in self.rules:
            deps.setdefault(rule.head, set()).update(rule.body)

        def has_cycle(lit, path):
            if lit in path:                    # we came back to a literal of the path
                return True
            for d in deps.get(lit, []):        # follow each dependency
                if has_cycle(d, path | {lit}):
                    return True
            return False

        return any(has_cycle(h, set()) for h in deps)

    def display(self):
        print("literals    :", sorted(self.literals))
        print("assumptions :", sorted(self.assumptions))
        print("rules       :")
        for rule in self.rules:
            print("   ", rule)
        print("contraries  :")
        for a, c in self.contraries.items():
            print(f"    contrary({a}) = {c}")
        print("preferences :", sorted(self.preferences))
        print("atomic      :", self.is_atomic())
        print("circular    :", self.is_circular())


def fmt(arg):
    # print an argument like {a} |- p
    X, s = arg
    return "{" + ",".join(sorted(X)) + "} |- " + s

## 2. Example of TD4 (circular and not atomic)
A = {a, b}, contrary(a) = y, contrary(b) = x, rules: y <- b, y <- y, x <- x, x <- a, z <- x, y.

In [2]:
f = ABAFramework()

# L
f.add_literal("x")
f.add_literal("y")
f.add_literal("z")

# A + contraries
f.add_assumption("a", "y")
f.add_assumption("b", "x")

# R
f.add_rule("y", ["b"])
f.add_rule("y", ["y"]) # cycle
f.add_rule("x", ["x"]) # cycle
f.add_rule("x", ["a"])
f.add_rule("z", ["x", "y"])

# framework with its properties
print("=== original framework ===")
f.display()

=== original framework ===
literals    : ['a', 'b', 'x', 'y', 'z']
assumptions : ['a', 'b']
rules       :
    y <- b
    y <- y
    x <- x
    x <- a
    z <- x, y
contraries  :
    contrary(a) = y
    contrary(b) = x
preferences : []
atomic      : False
circular    : True


### Conversion

In [3]:
print("=== to non circular ===")
f.to_non_circular().display()
print("\n=== after conversion (non circular + atomic) ===")
f.convert().display()

=== to non circular ===
literals    : ['a', 'b', 'x', 'x_1', 'x_2', 'y', 'y_1', 'y_2', 'z', 'z_1', 'z_2']
assumptions : ['a', 'b']
rules       :
    y_1 <- b
    y_2 <- b
    y <- b
    y_2 <- y_1
    y <- y_2
    x_2 <- x_1
    x <- x_2
    x_1 <- a
    x_2 <- a
    x <- a
    z_2 <- x_1, y_1
    z <- x_2, y_2
contraries  :
    contrary(a) = y
    contrary(b) = x
preferences : []
atomic      : False
circular    : False

=== after conversion (non circular + atomic) ===
literals    : ['a', 'b', 'x', 'x_1', 'x_1_d', 'x_1_nd', 'x_2', 'x_2_d', 'x_2_nd', 'x_d', 'x_nd', 'y', 'y_1', 'y_1_d', 'y_1_nd', 'y_2', 'y_2_d', 'y_2_nd', 'y_d', 'y_nd', 'z', 'z_1', 'z_1_d', 'z_1_nd', 'z_2', 'z_2_d', 'z_2_nd', 'z_d', 'z_nd']
assumptions : ['a', 'b', 'x_1_d', 'x_1_nd', 'x_2_d', 'x_2_nd', 'x_d', 'x_nd', 'y_1_d', 'y_1_nd', 'y_2_d', 'y_2_nd', 'y_d', 'y_nd', 'z_1_d', 'z_1_nd', 'z_2_d', 'z_2_nd', 'z_d', 'z_nd']
rules       :
    y_1 <- b
    y_2 <- b
    y <- b
    y_2 <- y_1_d
    y <- y_2_d
    x_2 <- x_1_d
 

### Arguments and attacks

In [4]:
args = f.build_arguments()
print("arguments :")
for a in sorted(fmt(x) for x in args):
    print("   ", a)
print("attacks :", len(f.build_attacks(args)))

# like the td correction
nc = f.to_non_circular()
args_nc = nc.build_arguments()
print("\narguments (non circular) :", len(args_nc))
for a in sorted(fmt(x) for x in args_nc):
    print("   ", a)
print("attacks (non circular) :", len(nc.build_attacks(args_nc)))

# atomic framework
cv = f.convert()
args_cv = cv.build_arguments()
print("\narguments (atomic) :", len(args_cv))
print("attacks (atomic) :", len(cv.build_attacks(args_cv)))

arguments :
    {a,b} |- z
    {a} |- a
    {a} |- x
    {b} |- b
    {b} |- y
attacks : 6

arguments (non circular) : 10
    {a,b} |- z
    {a,b} |- z_2
    {a} |- a
    {a} |- x
    {a} |- x_1
    {a} |- x_2
    {b} |- b
    {b} |- y
    {b} |- y_1
    {b} |- y_2
attacks (non circular) : 12

arguments (atomic) : 32
attacks (atomic) : 45


## 3. Preferences: the umbrella example
The forecast says rain and the sky looks sunny. We trust the forecast more than our eyes: sky_ok < forecast_ok.

In [ ]:
u = ABAFramework()
for lit in ["rain", "sun", "umbrella"]:
    u.add_literal(lit)
u.add_assumption("forecast_ok", "sun") # forecast is right, contrary = sun
u.add_assumption("sky_ok", "rain") # my eyes are right, contrary = rain
u.add_rule("rain", ["forecast_ok"])
u.add_rule("sun", ["sky_ok"])
u.add_rule("umbrella", ["rain"])
u.add_preference("sky_ok", "forecast_ok") # sky_ok < forecast_ok

u.display()
args = u.build_arguments()
print("\nattacks without preferences :", len(u.build_attacks(args)))
normal, reverse = u.build_attacks_plus(args)
print("normal attacks :", len(normal))
for x, y in sorted((fmt(x), fmt(y)) for x, y in normal):
    print("   ", x, "-->", y)
print("reverse attacks :", len(reverse))
for x, y in sorted((fmt(x), fmt(y)) for x, y in reverse):
    print("   ", x, "-->", y)

literals    : ['forecast_ok', 'rain', 'sky_ok', 'sun', 'umbrella']
assumptions : ['forecast_ok', 'sky_ok']
rules       :
    rain <- forecast_ok
    sun <- sky_ok
    umbrella <- rain
contraries  :
    contrary(forecast_ok) = sun
    contrary(sky_ok) = rain
preferences : [('sky_ok', 'forecast_ok')]
atomic      : False
circular    : False

attacks without preferences : 5
normal attacks : 2
    {forecast_ok} |- rain --> {sky_ok} |- sky_ok
    {forecast_ok} |- rain --> {sky_ok} |- sun
reverse attacks : 3
    {forecast_ok} |- forecast_ok --> {sky_ok} |- sun
    {forecast_ok} |- rain --> {sky_ok} |- sun
    {forecast_ok} |- umbrella --> {sky_ok} |- sun


## 4. Example of the online application (question 2)
Same framework as the example written in the application, with `PREF: a > b`.

In [7]:
g = ABAFramework()
for lit in ["a", "b", "c", "q", "p", "r", "s", "t"]:
    g.add_literal(lit)
g.add_assumption("a", "r")
g.add_assumption("b", "s")
g.add_assumption("c", "t")
g.add_rule("p", ["q", "a"])
g.add_rule("q", [])
g.add_rule("r", ["b", "c"])
g.add_rule("t", ["p", "c"])
g.add_rule("s", ["t"])
g.add_preference("b", "a")   # PREF: a > b

g.display()
args = g.build_arguments()
print("\narguments :", len(args))
print("attacks :", len(g.build_attacks(args)))
normal, reverse = g.build_attacks_plus(args)
print("normal attacks :", len(normal))
print("reverse attacks :", len(reverse))

cv = g.convert()
print("\nafter conversion : rules", len(cv.rules), ", assumptions", len(cv.assumptions))
print("arguments (atomic) :", len(cv.build_arguments()))

literals    : ['a', 'b', 'c', 'p', 'q', 'r', 's', 't']
assumptions : ['a', 'b', 'c']
rules       :
    p <- q, a
    q <-
    r <- b, c
    t <- p, c
    s <- t
contraries  :
    contrary(a) = r
    contrary(b) = s
    contrary(c) = t
preferences : [('b', 'a')]
atomic      : False
circular    : False

arguments : 8
attacks : 10
normal attacks : 6
reverse attacks : 4

after conversion : rules 22 , assumptions 53
arguments (atomic) : 75
